# 01 — Smoke Test (PGR207 CIFAR-10 mid-term)

**Student IDs:** `<STUDENT_ID_1>`, `<STUDENT_ID_2>` *(fill in before submitting)*

**Purpose:** verify the entire pipeline — data loading, the stratified split, all 3
augmentation strategies, all 3 architectures, all 3 optimizers, the training/early-stopping
loop, and metric computation — runs end to end **without errors**, in a few minutes, before
committing hours of Colab GPU time to the real 21-run experiment in
`02_full_experiment.ipynb`.

This notebook trains all **7 OFAT configurations** (Section 3.5 of the assignment brief) on a
small subset of the training data for 1–2 epochs. **The numbers produced here are not
meaningful results** — they only prove the pipeline works. Do not report them in the paper.

Everything reusable (data splitting, transforms, model builders, the training loop, metrics,
the `CONFIGS` dict) lives in the shared pipeline, split by concern across
`pipeline_config.py`, `pipeline_data.py`, `pipeline_models.py`, `pipeline_train.py`, and
`pipeline_metrics.py`. Both group members import from these same files.


## 1. Setup

In [1]:
# If running on Colab, pipeline_config.py, pipeline_data.py, pipeline_models.py,
# pipeline_train.py, and pipeline_metrics.py must be in the same directory as this
# notebook (e.g. upload them, or mount/clone the shared repo folder first).
import pipeline_config as cfg
import pipeline_data as data
import pipeline_models as models
import pipeline_train as train

print("Device:", data.DEVICE)


Device: mps


In [2]:
from pipeline_models import get_param_counts
for name, n in get_param_counts().items():
    print(f"{name}: {n:,}")

simple_cnn: 620,362
resnet18: 11,173,962
densenet121: 6,956,426


## 2. Load data and build the fixed stratified split

This is the **same split** (`random_state=42`, cached to `val_split_indices.npz`) that
`02_full_experiment.ipynb` will use — created once here (or loaded from cache if it already
exists) and reused everywhere, per Section 4 of the brief.


In [ ]:
full_train, test_set = data.load_raw_datasets()
train_idx, val_idx = data.get_stratified_split(full_train.targets)

print(f"Train pool: {len(train_idx)}  Val: {len(val_idx)}  Test (held out): {len(test_set)}")


## 3. Sanity-check the 7 configurations and architecture parameter counts

Every row of `CONFIGS` should differ from the baseline `C1` in exactly one factor — this is
the self-check the brief recommends (Section 3.5) before spending any compute.


In [ ]:
for cid, config in cfg.CONFIGS.items():
    print(cid, config)

print()
print("Parameter counts:")
for name, n in models.get_param_counts().items():
    print(f"  {name}: {n:,}")


## 4. Run all 7 configs at reduced scale

Settings used here (`SMOKE_SUBSET_SIZE`, `SMOKE_EPOCHS`, `SMOKE_PATIENCE`) are defined in
`pipeline_config.py` so they can't silently drift from what both group members run. Only
**seed 0** is used — the smoke test does not need all 3 seeds, it just needs to prove the
code path works.


In [ ]:
smoke_results = []

for cid, config in cfg.CONFIGS.items():
    print(f"--- Smoke test: {cid} {config} ---")
    result = train.run_training(
        config_id=cid,
        config=config,
        seed=0,
        full_train=full_train,
        test_set=test_set,
        train_idx=train_idx,
        val_idx=val_idx,
        epochs=cfg.SMOKE_EPOCHS,
        patience=cfg.SMOKE_PATIENCE,
        batch_size=cfg.BATCH_SIZE,
        subset_size=cfg.SMOKE_SUBSET_SIZE,
        verbose=True,
    )
    smoke_results.append(result)
    print(f"  -> test_acc={result['test_accuracy']:.3f}  macro_f1={result['test_macro_f1']:.3f}  "
          f"({result['train_time_seconds']:.1f}s)")


## 5. Inspect results

A quick tabular view — confirms every config produced a row with sane-looking metrics.

In [ ]:
import pandas as pd

smoke_df = pd.DataFrame(smoke_results)
smoke_df[[
    "config_id", "architecture", "augmentation", "optimizer",
    "test_accuracy", "test_macro_f1", "epochs_run", "early_stopped", "train_time_seconds",
]]


## 6. Checklist before moving to the full experiment

- [ ] All 7 configs ran without errors.
- [ ] Accuracy is well above chance (10%) for every config — on this tiny subset it will be
      low (subset is only 1000 images, 1-2 epochs), but not near-random.
- [ ] `val_split_indices.npz` was created in this directory — confirm it exists and will be
      reused (not regenerated) by `02_full_experiment.ipynb`.
- [ ] Parameter counts look sane relative to each other (SimpleCNN << DenseNet-121 approx ResNet-18,
      exact ordering depends on the adaptation).

If all of the above hold, proceed to `02_full_experiment.ipynb` for the real 21-run experiment.
